In [ ]:
import json
import os
import re
import sys
import time
import importlib
from pathlib import Path

import torch
from dotenv import load_dotenv
from pymilvus import MilvusClient, DataType

from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableStructureOptions,
    TableFormerMode,
    EasyOcrOptions,
    LayoutOptions,
    AcceleratorOptions,
    AcceleratorDevice,
)
from docling.datamodel.layout_model_specs import DOCLING_LAYOUT_EGRET_XLARGE
from docling.chunking import HybridChunker

In [ ]:
BASE_DIR = Path(r"D:\Final_GRAG")
sys.path.insert(0, str(BASE_DIR))

REPORTS_DIR = BASE_DIR / "reports"

import src.report_chunk_cleaner as _cleaner_mod
import src.embedding_utils as _emb_mod

In [ ]:
USE_CUDA = torch.cuda.is_available()
DEVICE_STR = "cuda:0" if USE_CUDA else "cpu"
print(f"CUDA available: {USE_CUDA} | device = {DEVICE_STR}")
if USE_CUDA:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
def pdf_to_report_id(pdf_path):
    file_stem = pdf_path.stem
    if "_Sustainability_report_" in file_stem:
        name, year = file_stem.split("_Sustainability_report_", 1)
        return name.replace("_", "") + year
    return file_stem.replace("_", "")


pdf_files = sorted(REPORTS_DIR.glob("*.pdf"))
print(f"Tổng số reports: {len(pdf_files)}")

## Cấu hình DocumentConverter và Chunker

In [ ]:
accelerator_options = AcceleratorOptions(
    num_threads=8,
    device=AcceleratorDevice.CUDA if USE_CUDA else AcceleratorDevice.CPU,
)

pipeline_options = PdfPipelineOptions(
    accelerator_options=accelerator_options,

    do_ocr=True,
    ocr_options=EasyOcrOptions(
        lang=["en"],
        confidence_threshold=0.60,
        force_full_page_ocr=False,
    ),

    layout_options=LayoutOptions(
        model_spec=DOCLING_LAYOUT_EGRET_XLARGE,
        create_orphan_clusters=False,
        keep_empty_clusters=False,
    ),

    # text-only pipeline
    do_table_structure=False,
    generate_picture_images=False,
    do_picture_classification=False,
    do_picture_description=False,
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

chunker = HybridChunker(
    tokenizer="BAAI/bge-m3",
    max_tokens=700,
    merge_peers=True,
)

## Xử lý batch: convert + clean + lưu chunks

In [ ]:
importlib.reload(_cleaner_mod)
from src.report_chunk_cleaner import build_clean_chunks

OUTPUT_DIR = BASE_DIR / "metadata" / "report_units"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Lọc theo report_id (để trống → chạy hết)
PROCESS_REPORT_IDS = []
batch_files = pdf_files
if PROCESS_REPORT_IDS:
    wanted = set(PROCESS_REPORT_IDS)
    batch_files = [p for p in pdf_files if pdf_to_report_id(p) in wanted]

print(f"Số reports sẽ xử lý: {len(batch_files)}")
print(f"Output dir: {OUTPUT_DIR}")

for i, pdf_path in enumerate(batch_files, 1):
    report_id = pdf_to_report_id(pdf_path)
    report_dir = OUTPUT_DIR / report_id
    report_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[{i}/{len(batch_files)}] Đang xử lý: {pdf_path.name}")
    t0 = time.time()
    result = converter.convert(str(pdf_path))
    convert_elapsed = time.time() - t0

    doc = result.document
    num_pages = len(doc.pages) if hasattr(doc, "pages") else 0

    t1 = time.time()
    cleaned_chunks, cleaning_report = build_clean_chunks(
        doc=doc,
        chunker=chunker,
        report_id=report_id,
    )
    chunk_elapsed = time.time() - t1

    report_chunks_path = report_dir / "report_chunks.json"
    with open(report_chunks_path, "w", encoding="utf-8") as f:
        json.dump(cleaned_chunks, f, ensure_ascii=False, indent=2)

    cleaning_report_path = report_dir / "_cleaning_report.json"
    with open(cleaning_report_path, "w", encoding="utf-8") as f:
        json.dump(cleaning_report, f, ensure_ascii=False, indent=2)

    raw_n = cleaning_report["raw_chunk_count"]
    final_n = cleaning_report["final_chunk_count"]
    drops_str = ", ".join(
        f"{stage}={count}" for stage, count in cleaning_report["stage_drops"].items()
    ) or "không có"
    print(f"--- Convert {convert_elapsed:.1f}s | Chunk+Clean {chunk_elapsed:.1f}s ---")
    print(f"Số trang      : {num_pages}")
    print(f"Raw chunks    : {raw_n}")
    print(f"Final chunks  : {final_n}  (giữ {final_n / raw_n:.0%})" if raw_n else f"Final chunks  : {final_n}")
    print(f"Stage drops   : {drops_str}")
    print(f"Saved chunks  : {report_chunks_path}")
    print(f"Saved cleaning: {cleaning_report_path}")

## Kết nối Zilliz và encoder BGE-M3

In [ ]:
importlib.reload(_emb_mod)
from src.embedding_utils import BGEM3Encoder

load_dotenv(BASE_DIR / ".env")
ZILLIZ_URI = os.getenv("ZILLIZ_CLOUD_URI")
ZILLIZ_KEY = os.getenv("ZILLIZ_CLOUD_API_KEY")

milvus_client = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_KEY)
encoder = BGEM3Encoder(use_fp16=True, devices=DEVICE_STR)

REPORT_CHUNKS_COLLECTION = "report_chunks"
print(f"Milvus client connected: {ZILLIZ_URI}")
print(f"Encoder device: {encoder.devices}")

## Tạo collection report_chunks

In [ ]:
# Schema khớp với report_chunks.json sinh từ build_clean_chunks.
# Các trường list (page_numbers, heading_path, doc_item_labels) lưu dạng string serialize
# để HybridRetriever có thể filter bằng like "%N%".
# Drop-and-recreate cho phép chạy lại cell nhiều lần (idempotent).

_VARCHAR_FIELDS = {
    "content_text":   65535,
    "report_id":      64,
    "modality":       32,
    "chunk_type":     32,
    "page_numbers":   256,
    "heading_path":   2048,
    "section_label":  1024,
    "doc_item_labels": 512,
}

schema = MilvusClient.create_schema(auto_id=False, enable_dynamic_field=False)

schema.add_field("chunk_id",         DataType.VARCHAR,        max_length=128, is_primary=True)
schema.add_field("dense_embedding",  DataType.FLOAT_VECTOR,   dim=1024)
schema.add_field("sparse_embedding", DataType.SPARSE_FLOAT_VECTOR)

for _fname, _maxlen in _VARCHAR_FIELDS.items():
    schema.add_field(_fname, DataType.VARCHAR, max_length=_maxlen)

for _ifield in ("chunk_index", "page_start", "page_end", "char_count", "token_count"):
    schema.add_field(_ifield, DataType.INT32)

index_params = milvus_client.prepare_index_params()
index_params.add_index("chunk_id")
index_params.add_index("dense_embedding",  index_type="AUTOINDEX",             metric_type="COSINE")
index_params.add_index("sparse_embedding", index_type="SPARSE_INVERTED_INDEX", metric_type="IP")
index_params.add_index("report_id")
index_params.add_index("chunk_type")

if REPORT_CHUNKS_COLLECTION in milvus_client.list_collections():
    milvus_client.drop_collection(REPORT_CHUNKS_COLLECTION)
    print(f"Dropped existing collection : {REPORT_CHUNKS_COLLECTION}")

milvus_client.create_collection(
    collection_name=REPORT_CHUNKS_COLLECTION,
    schema=schema,
    index_params=index_params,
)
print(f"Created collection          : {REPORT_CHUNKS_COLLECTION}")
print(f"Fields                      : {[f.name for f in schema.fields]}")

## Embed và upload chunks lên Zilliz

In [ ]:
REPORT_UNITS_DIR = BASE_DIR / "metadata" / "report_units"
EMBED_BATCH  = 32   # số chunks mỗi lần encode
INSERT_BATCH = 100  # số dòng mỗi lần insert vào Milvus

# Độ dài tối đa VARCHAR phải khớp với schema collection (cell tạo collection)
VARCHAR_MAX = {
    "chunk_id":        128,
    "content_text":    65535,
    "report_id":       64,
    "modality":        32,
    "chunk_type":      32,
    "page_numbers":    256,
    "heading_path":    2048,
    "section_label":   1024,
    "doc_item_labels": 512,
}
INT_FIELDS = ("chunk_index", "page_start", "page_end", "char_count", "token_count")


def prepare_chunk(chunk):
    """Chuyển 1 chunk đã embed thành record sẵn sàng insert vào Zilliz."""
    rec = {}
    for field, max_len in VARCHAR_MAX.items():
        if field in ("page_numbers", "heading_path", "doc_item_labels"):
            raw = chunk.get(field)
            val = str(raw) if raw is not None else "[]"
        else:
            val = str(chunk.get(field) or "")
        rec[field] = val[:max_len]

    for field in INT_FIELDS:
        v = chunk.get(field)
        rec[field] = int(v) if v is not None else 0

    rec["dense_embedding"]  = chunk["dense_embedding"]
    rec["sparse_embedding"] = chunk["sparse_embedding"]
    return rec


chunk_files = sorted(REPORT_UNITS_DIR.glob("*/report_chunks.json"))
print(f"Tìm thấy {len(chunk_files)} file report_chunks.json\n")
print(f"{'Report':<25}  {'Chunks':>6}  {'Thời gian':>9}")
print("-" * 45)

total_inserted = 0
for chunk_file in chunk_files:
    report_id = chunk_file.parent.name
    t0 = time.time()

    with open(chunk_file, encoding="utf-8") as f:
        chunks = json.load(f)

    # Embed: thêm dense_embedding + sparse_embedding vào mỗi chunk
    chunks = encoder.encode_chunks(chunks, batch_size=EMBED_BATCH)

    # Chuẩn bị record để insert
    prepared = [prepare_chunk(c) for c in chunks]

    # Insert theo batch
    inserted = 0
    for i in range(0, len(prepared), INSERT_BATCH):
        batch = prepared[i : i + INSERT_BATCH]
        milvus_client.insert(collection_name=REPORT_CHUNKS_COLLECTION, data=batch)
        inserted += len(batch)

    total_inserted += inserted
    elapsed = time.time() - t0
    print(f"{report_id:<25}  {inserted:>6}  {elapsed:>8.1f}s")

print("-" * 45)
print(f"{'TỔNG':<25}  {total_inserted:>6}")
print(f"\nĐã upload tất cả chunks lên collection '{REPORT_CHUNKS_COLLECTION}'.")